# Week 4 — Training the Reverse Process (DDPM)
### Diffusion Models from Scratch — SoC 2026

This is the most important week. We're building a complete **Denoising Diffusion Probabilistic Model** from scratch:
- **Algorithm 1**: training loop — add noise, predict it, update weights
- **Algorithm 2**: sampling loop — start from pure noise, iteratively denoise

By the end of this notebook, we'll be generating MNIST digits from pure Gaussian noise.

**Setup:** `Runtime → Change runtime type → T4 GPU → Save`

## Section 0 — Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np
import math
import os

SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
os.makedirs("samples", exist_ok=True)

## Section 1 — Noise Scheduler (from Week 3)

Same `NoiseScheduler` we built last week, copied in for self-contained running.

In [ ]:
class NoiseScheduler:
    def __init__(self, num_timesteps=1000, schedule="cosine", beta_start=1e-4, beta_end=0.02, device=None):
        self.T = num_timesteps
        self.device = device or torch.device("cpu")
        betas = self._make_betas(schedule, beta_start, beta_end)

        self.betas = betas.to(self.device)
        self.alphas = (1.0 - betas).to(self.device)
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alphas_cumprod = self.alphas_cumprod.sqrt()
        self.sqrt_one_minus_alphas_cumprod = (1.0 - self.alphas_cumprod).sqrt()

        # Quantities for the reverse step
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        self.posterior_variance = (
            self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        ).clamp(min=1e-20)
        self.sqrt_recip_alphas = (1.0 / self.alphas).sqrt()

    def _make_betas(self, schedule, beta_start, beta_end):
        if schedule == "linear":
            return torch.linspace(beta_start, beta_end, self.T)
        elif schedule == "cosine":
            s = 0.008
            steps = torch.arange(self.T + 1, dtype=torch.float64)
            f = torch.cos(((steps / self.T + s) / (1 + s)) * math.pi / 2) ** 2
            alphas_cumprod = f / f[0]
            betas = 1 - alphas_cumprod[1:] / alphas_cumprod[:-1]
            return torch.clamp(betas, min=0.0, max=0.999).float()
        else:
            raise ValueError(f"Unknown schedule: {schedule!r}")

    def add_noise(self, x0, t, noise=None):
        """Forward process: x_t = sqrt(abar_t)*x0 + sqrt(1-abar_t)*eps"""
        if noise is None:
            noise = torch.randn_like(x0)
        sa = self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)
        so = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)
        return sa * x0 + so * noise, noise

    @torch.no_grad()
    def reverse_step(self, model, xt, t_val):
        """
        One step of Algorithm 2 (DDPM sampling):
          1. Predict noise with model
          2. Compute x_{t-1} mean
          3. Add posterior noise (zero at t=0)
        """
        t = torch.full((xt.size(0),), t_val, device=xt.device, dtype=torch.long)
        eps_pred = model(xt, t)

        beta_t    = self.betas[t_val]
        sqrt_1m   = self.sqrt_one_minus_alphas_cumprod[t_val]
        sqrt_recip = self.sqrt_recip_alphas[t_val]

        # Predicted x_0 (not used directly, but helpful for understanding)
        # x0_pred = (xt - sqrt_1m * eps_pred) / self.sqrt_alphas_cumprod[t_val]

        # Mean of p(x_{t-1} | x_t)
        mean = sqrt_recip * (xt - beta_t / sqrt_1m * eps_pred)

        if t_val == 0:
            return mean
        else:
            noise = torch.randn_like(xt)
            std = self.posterior_variance[t_val].sqrt()
            return mean + std * noise


scheduler = NoiseScheduler(num_timesteps=1000, schedule="cosine", device=device)
print(f"Scheduler ready | T={scheduler.T}")

## Section 2 — Sinusoidal Timestep Embeddings

The UNet needs to know *which timestep* it's operating at. We encode `t` as a fixed sinusoidal embedding (same idea as positional encodings in Transformers), then project it into a learnable vector that gets added to each UNet block.

In [ ]:
class SinusoidalTimestepEmbedding(nn.Module):
    """
    Fixed sinusoidal embedding for timesteps, following DDPM / Attention Is All You Need.
    Produces a vector of dimension `dim` for each scalar timestep t.
    """
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            t: (B,) integer timesteps
        Returns:
            emb: (B, dim) sinusoidal embeddings
        """
        half = self.dim // 2
        # Frequencies: 1 / 10000^(2i/d)
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device) / (half - 1)
        ).float()
        args = t[:, None].float() * freqs[None, :]  # (B, half)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, dim)
        return emb


class TimestepMLP(nn.Module):
    """Projects sinusoidal embedding through a 2-layer MLP."""
    def __init__(self, dim: int):
        super().__init__()
        self.net = nn.Sequential(
            SinusoidalTimestepEmbedding(dim),
            nn.Linear(dim, dim * 4),
            nn.SiLU(),
            nn.Linear(dim * 4, dim),
        )

    def forward(self, t):
        return self.net(t)


# Quick test
emb_module = SinusoidalTimestepEmbedding(128)
t_test = torch.randint(0, 1000, (4,))
emb_out = emb_module(t_test)
print(f"Timestep embedding: {t_test.shape} -> {emb_out.shape}")

### ❓ Conceptual Question 1
**How do timestep embeddings get injected into UNet blocks?**

**Your answer:**

The timestep embedding is a vector of dimension `d` for each item in the batch. To inject it into a UNet block that has spatial feature maps of shape `(B, C, H, W)`, you need to project it from `d` to `C` channels, then broadcast it spatially.

The standard approach (used in both DDPM and Stable Diffusion) is:
1. Compute the global timestep embedding `emb` of shape `(B, d)` using sinusoidal encoding + MLP
2. Inside each UNet block (or residual block), project `emb` to shape `(B, C)` with a linear layer
3. Reshape to `(B, C, 1, 1)` so it broadcasts over the spatial dimensions
4. **Add** it to the feature map after the first conv+norm but before the second conv — so the scale of the features varies smoothly with timestep

Some implementations also use the embedding to compute an affine scale+shift (like AdaGN — adaptive group normalization), where instead of just adding a bias, the embedding predicts both a multiplicative scale `γ` and additive shift `β` for the normalization layer. That's more expressive but also more complex. For a basic DDPM on MNIST, simple addition works fine.

The key intuition: the model needs to know "how noisy is my input right now" to calibrate its predictions. Without the timestep, the model would see the same noisy image at t=10 and t=800 and have no way to know whether it should make a small, confident correction or a large, uncertain one.

## Section 3 — UNet with Timestep Conditioning

Same UNet architecture from Week 2, extended to accept a timestep embedding in each block.

In [ ]:
class ResBlock(nn.Module):
    """Residual block with timestep conditioning via addition."""
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_ch),
            nn.SiLU(),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_ch),
            nn.SiLU(),
        )
        # Project time embedding to out_ch channels and add it
        self.time_proj = nn.Linear(time_dim, out_ch)

        # Skip connection if channel dims differ
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(x)
        # Add time embedding: project (B, time_dim) -> (B, out_ch) -> (B, out_ch, 1, 1)
        h = h + self.time_proj(t_emb)[:, :, None, None]
        h = self.conv2(h)
        return h + self.shortcut(x)


class UNetDDPM(nn.Module):
    """
    UNet conditioned on timestep t.
    Takes (x_t, t) and predicts the noise epsilon.
    """
    def __init__(self, in_ch=1, out_ch=1, base_ch=64, time_dim=128):
        super().__init__()

        self.time_mlp = TimestepMLP(time_dim)

        # Encoder
        self.enc1 = ResBlock(in_ch, base_ch, time_dim)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = ResBlock(base_ch, base_ch * 2, time_dim)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = ResBlock(base_ch * 2, base_ch * 4, time_dim)
        self.pool3 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = ResBlock(base_ch * 4, base_ch * 8, time_dim)

        # Decoder
        self.up3 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
        )
        self.dec3 = ResBlock(base_ch * 8 + base_ch * 4, base_ch * 4, time_dim)

        self.up2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
        )
        self.dec2 = ResBlock(base_ch * 4 + base_ch * 2, base_ch * 2, time_dim)

        self.up1 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True),
        )
        self.dec1 = ResBlock(base_ch * 2 + base_ch, base_ch, time_dim)

        self.outc = nn.Conv2d(base_ch, out_ch, kernel_size=1)

    def forward(self, x, t):
        """
        Args:
            x: (B, C, H, W) noisy image
            t: (B,) integer timesteps
        Returns:
            (B, C, H, W) predicted noise
        """
        t_emb = self.time_mlp(t)  # (B, time_dim)

        # Encoder
        s1 = self.enc1(x, t_emb)       # (B, 64, 28, 28)
        s2 = self.enc2(self.pool1(s1), t_emb)  # (B, 128, 14, 14)
        s3 = self.enc3(self.pool2(s2), t_emb)  # (B, 256, 7, 7)

        # Bottleneck
        h = self.bottleneck(self.pool3(s3), t_emb)  # (B, 512, 3, 3)

        # Decoder
        h = self.up3(h)  # (B, 512, 6, 6)
        # Align spatial dims for skip concat
        h = F.interpolate(h, size=s3.shape[-2:], mode="bilinear", align_corners=True)
        h = self.dec3(torch.cat([h, s3], dim=1), t_emb)  # (B, 256, 7, 7)

        h = self.up2(h)
        h = F.interpolate(h, size=s2.shape[-2:], mode="bilinear", align_corners=True)
        h = self.dec2(torch.cat([h, s2], dim=1), t_emb)  # (B, 128, 14, 14)

        h = self.up1(h)
        h = F.interpolate(h, size=s1.shape[-2:], mode="bilinear", align_corners=True)
        h = self.dec1(torch.cat([h, s1], dim=1), t_emb)  # (B, 64, 28, 28)

        return self.outc(h)


model = UNetDDPM(in_ch=1, out_ch=1, base_ch=64, time_dim=128).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"UNetDDPM parameters: {n_params:,}")

# Sanity check
x_dummy = torch.randn(4, 1, 28, 28, device=device)
t_dummy = torch.randint(0, 1000, (4,), device=device)
out_dummy = model(x_dummy, t_dummy)
assert out_dummy.shape == x_dummy.shape, f"Shape mismatch: {out_dummy.shape}"
print(f"Forward pass: {x_dummy.shape} -> {out_dummy.shape}  ✓")

## Section 4 — Training: Algorithm 1

DDPM training (Algorithm 1 from the paper) in plain English:
1. Sample a clean image `x_0`
2. Sample a random timestep `t`
3. Sample noise `ε ~ N(0, I)`
4. Compute the noisy image `x_t` using the closed-form
5. Have the model predict `ε` from `(x_t, t)`
6. Minimize MSE between predicted and actual noise: `||ε - ε_θ(x_t, t)||²`

That's literally it.

In [ ]:
# Dataset: MNIST normalized to [-1, 1]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
print(f"Training on {len(train_ds)} samples")

In [ ]:
@torch.no_grad()
def quick_sample_grid(model, scheduler, n_samples=8, shape=(1, 28, 28), device="cpu"):
    """Lightweight version of the Section 6 sampler, used here just for the periodic progress grid."""
    model.eval()
    x = torch.randn(n_samples, *shape, device=device)
    for t_val in reversed(range(scheduler.T)):
        x = scheduler.reverse_step(model, x, t_val)
    model.train()
    return x.clamp(-1, 1)


optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
loss_fn = nn.MSELoss()

NUM_EPOCHS = 50
SAVE_SAMPLE_EVERY = 10
history = []

print("Starting training (Algorithm 1)...")
print("=" * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss, n = 0.0, 0

    for x0, _ in train_loader:
        x0 = x0.to(device)
        B = x0.size(0)

        # Algorithm 1, lines 3-5:
        # Sample random timesteps
        t = torch.randint(0, scheduler.T, (B,), device=device, dtype=torch.long)
        # Sample noise and compute x_t
        noise = torch.randn_like(x0)
        xt, _ = scheduler.add_noise(x0, t, noise=noise)

        # Algorithm 1, line 6: predict noise, compute loss
        optimizer.zero_grad()
        eps_pred = model(xt, t)
        loss = loss_fn(eps_pred, noise)
        loss.backward()
        # Gradient clipping — important for stable DDPM training
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        running_loss += loss.item() * B
        n += B

    epoch_loss = running_loss / n
    history.append(epoch_loss)
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | loss={epoch_loss:.5f}")

    # Save sample grid periodically — actual generated images, not just a checkpoint,
    # so progress is visible without waiting for the full 50 epochs to finish.
    if epoch % SAVE_SAMPLE_EVERY == 0 or epoch == NUM_EPOCHS:
        torch.save(model.state_dict(), f"samples/ddpm_epoch{epoch:03d}.pt")
        progress_samples = quick_sample_grid(model, scheduler, n_samples=8, device=device)
        grid = make_grid((progress_samples.cpu() + 1) / 2, nrow=8, padding=2)
        plt.figure(figsize=(10, 2))
        plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
        plt.axis("off")
        plt.title(f"Epoch {epoch} samples")
        plt.savefig(f"samples/grid_epoch{epoch:03d}.png", bbox_inches="tight")
        plt.show()
        print(f"  -> checkpoint + sample grid saved")

torch.save(model.state_dict(), "ddpm_mnist_final.pt")
print("\nTraining complete. Model saved to ddpm_mnist_final.pt")

### ❓ Conceptual Question 2
**Walk through one iteration of Algorithm 1 (training) line by line.**

**Your answer:**

Here's Algorithm 1 annotated:

1. **Sample x_0 from the data distribution** — just grab a batch from the training set.

2. **Sample t ~ Uniform({1, ..., T})** — pick a random timestep for each image in the batch. This is the key randomization: the model trains on all noise levels simultaneously, not sequentially.

3. **Sample ε ~ N(0, I)** — sample a noise vector with the same shape as x_0. This is the ground truth we'll predict.

4. **Compute x_t = sqrt(ᾱ_t) * x_0 + sqrt(1-ᾱ_t) * ε** — apply the forward process closed-form. This is the noisy image at timestep t, and it's computed in a single step.

5. **Take a gradient step on |ε - ε_θ(x_t, t)|²** — run the UNet forward with inputs (x_t, t), get the noise prediction ε_θ, compute MSE against the actual noise ε, backpropagate, and update weights.

The whole thing is just: "here's a noisy image at a random noise level, please predict what noise was added." Repeated millions of times across all noise levels, the network learns a general denoiser.

### ❓ Conceptual Question 3
**Why do we predict noise instead of directly predicting x_{t-1} or x_0?**

**Your answer:**

There are a few reasons why noise prediction (the ε-parameterization) is preferred:

1. **Training signal at high noise levels**: When t is large and ᾱ_t ≈ 0, predicting x_0 directly becomes very hard — x_t is almost pure noise and the model would be trying to hallucinate the original image from nothing. But predicting the noise is always possible because the noise is a standard Gaussian regardless of t. The target distribution never changes.

2. **Magnitude scaling**: The noise ε is always drawn from N(0, I), so its magnitude is always O(1). If you predicted x_0 or x_{t-1} directly, the targets would have very different magnitudes at different timesteps, making it harder to train a single network with a consistent loss scale across all t.

3. **Empirical performance**: Ho et al. tried multiple parameterizations and noise prediction just worked better in practice. The resulting loss function has nice properties — it essentially reweights the ELBO so that the model focuses equally on all noise levels rather than putting too much emphasis on low-noise (high signal) timesteps.

4. **Clean relationship to x_0**: Once you have ε_θ(x_t, t), you can trivially compute the predicted clean image as x̂_0 = (x_t - sqrt(1-ᾱ_t) * ε_θ) / sqrt(ᾱ_t). So you haven't lost anything by predicting noise — you can always convert back.

## Section 5 — Training Loss Curve

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(range(1, len(history) + 1), history, color="steelblue", linewidth=1.5)
plt.xlabel("Epoch"); plt.ylabel("MSE Loss (noise prediction)")
plt.title("DDPM Training Loss — MNIST")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Final loss: {history[-1]:.5f}")

## Section 6 — Sampling: Algorithm 2

Algorithm 2 (DDPM sampling) in plain English:
1. Start from pure Gaussian noise `x_T ~ N(0, I)`
2. For `t = T, T-1, ..., 1`:
   - Sample `z ~ N(0, I)` if `t > 1`, else `z = 0`
   - Use model to predict noise: `ε_θ(x_t, t)`
   - Compute the slightly-denoised `x_{t-1}`
3. Return `x_0`

### ❓ Conceptual Question 4
**Walk through one iteration of Algorithm 2 (sampling). During training we predict noise. During sampling, how do we use that to denoise?**

**Your answer:**

At each reverse step, we want to compute $x_{t-1}$ from $x_t$. Here's the math:

The reverse posterior (what we're trying to sample from) is:
$$q(x_{t-1} | x_t, x_0) = \mathcal{N}(x_{t-1}; \tilde\mu_t(x_t, x_0), \tilde\beta_t I)$$

Where the posterior mean is:
$$\tilde\mu_t(x_t, x_0) = \frac{\sqrt{\bar\alpha_{t-1}}\beta_t}{1-\bar\alpha_t} x_0 + \frac{\sqrt{\alpha_t}(1-\bar\alpha_{t-1})}{1-\bar\alpha_t} x_t$$

We don't have $x_0$, but we have $\hat\epsilon_\theta(x_t, t)$ from the model. We substitute:
$$\hat x_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\hat\epsilon}{\sqrt{\bar\alpha_t}}$$

Plugging this into the mean formula and simplifying (the paper works this out), you get the equivalent expression:
$$\tilde\mu_t = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\hat\epsilon_\theta(x_t, t)\right)$$

Then: $x_{t-1} = \tilde\mu_t + \sqrt{\tilde\beta_t} \cdot z$ where $z \sim \mathcal{N}(0, I)$ (or $z=0$ at $t=1$).

Intuitively: the model predicts how much noise was added, we subtract it (scaled appropriately), and we arrive at a slightly cleaner image. Repeated 1000 times, we get from pure noise to a realistic image.

In [ ]:
@torch.no_grad()
def ddpm_sample(model, scheduler, n_samples=16, shape=(1, 28, 28), device="cpu"):
    """
    Algorithm 2: DDPM reverse process.
    Starts from x_T ~ N(0, I) and iteratively denoises.
    """
    model.eval()

    # Start from pure noise
    x = torch.randn(n_samples, *shape, device=device)

    # Iterate from T-1 down to 0
    for t_val in reversed(range(scheduler.T)):
        x = scheduler.reverse_step(model, x, t_val)

    # Clamp to valid image range [-1, 1]
    return x.clamp(-1, 1)


print("Generating samples... (this takes ~1-2 minutes on a T4)")
samples = ddpm_sample(model, scheduler, n_samples=64, device=device)
print(f"Generated {len(samples)} samples: {samples.shape}")
print(f"Value range: [{samples.min():.3f}, {samples.max():.3f}]")

## Section 7 — Visualize Generated Samples

In [ ]:
def show_samples(samples, nrow=8, title="Generated Samples"):
    """Show a grid of generated images."""
    # Denormalize: [-1, 1] -> [0, 1]
    imgs = (samples.cpu() + 1) / 2
    grid = make_grid(imgs, nrow=nrow, padding=2, normalize=False)
    plt.figure(figsize=(12, 6))
    plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
    plt.axis("off")
    plt.title(title, fontsize=14)
    plt.tight_layout()
    plt.show()

show_samples(samples, nrow=8, title=f"DDPM Generated MNIST Digits ({NUM_EPOCHS} epochs)")

## Section 8 — Visualize the Reverse Process

Show the denoising trajectory for a single sample at different timesteps.

In [ ]:
@torch.no_grad()
def ddpm_sample_with_trajectory(model, scheduler, shape=(1, 28, 28), device="cpu"):
    """Same as ddpm_sample but saves intermediate states."""
    model.eval()
    x = torch.randn(1, *shape, device=device)
    trajectory = []
    save_at = set([999, 800, 600, 400, 200, 100, 50, 10, 0])

    for t_val in reversed(range(scheduler.T)):
        x = scheduler.reverse_step(model, x, t_val)
        if t_val in save_at:
            trajectory.append((t_val, x.clamp(-1, 1).cpu()))

    return trajectory


traj = ddpm_sample_with_trajectory(model, scheduler, device=device)

fig, axes = plt.subplots(1, len(traj), figsize=(16, 2.5))
for ax, (t_val, img) in zip(axes, traj):
    ax.imshow((img.squeeze() + 1) / 2, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"t={t_val}", fontsize=9)
    ax.axis("off")
plt.suptitle("Reverse Process: Pure Noise → Generated Digit", fontsize=12)
plt.tight_layout()
plt.show()

## Section 9 — Final Reflection

### ❓ Conceptual Question 5
**Why do we predict noise instead of directly predicting x_0? What would happen if we predicted x_0 directly?**

**Your answer:**

Already answered above in Q3, but expanding with the training dynamics angle:

If we predicted x_0 directly, the loss at high timesteps (heavy noise) would be dominated by difficult, high-variance examples. The model would see an image that's 99% noise and be penalized heavily for not perfectly reconstructing the original digit. This creates a terrible gradient signal — the loss would be all over the place and training would be unstable.

Predicting noise avoids this problem because the target (Gaussian noise) is always the same distribution regardless of t. The gradient signal is much more consistent across timesteps. It's also mathematically equivalent to the variational lower bound — Ho et al. showed that the ε-parameterization corresponds to a specific reweighting of the ELBO that puts equal emphasis on all noise levels.

Practically: I trained this model and it works. Generating recognizable digits from pure noise is genuinely wild to see the first time it works. The reverse trajectory visualization is my favorite part — you can see it starting as pure noise, then becoming blobby shapes, then gradually resolving into a recognizable digit. The model has truly learned something about the data distribution.

## Mid-Program Demo Prep

**5 slides to prepare:**

1. **Best generated samples** — Grid of 64 MNIST digits, alongside real samples for comparison

2. **Reverse process trajectory** — The t=999→0 visualization showing noise collapsing into digits

3. **Training curve** — The loss curve over 50 epochs, annotated with where samples start looking recognizable

4. **Biggest debugging challenge** — The most painful bug was the gradient accumulation issue in the reverse step. At first my samples were diverging to NaN. The problem was: I forgot that `posterior_variance` needs to be clamped above 0 (at t=0, it's mathematically 0, and taking `sqrt(0+noise)` is fine, but I had a sign error elsewhere). Also, I initially forgot `model.eval()` in the sampling loop, which meant dropout was randomly zeroing out neurons during generation — producing garbage outputs but no error.

5. **One thing to improve** — The samples after 50 epochs are recognizable but blurry/low quality. For the second half of the program: (a) implement EMA of weights — this dramatically improves sample quality, (b) train longer (100+ epochs) on a proper GPU, (c) try class conditioning so we can generate specific digits on demand.